In [11]:
import tensorflow as tf
import numpy as np
import pandas as pd
import sklearn

print(tf.__version__)
print(np.__version__)
print(sklearn.__version__)

2.21.0
2.4.4
1.8.0


In [4]:
df = pd.read_csv('yoga_pose.csv')
df.head()

,label,x0,y0,z0,v0,x1,y1,z1,v1,x2,...,z30,v30,x31,y31,z31,v31,x32,y32,z32,v32
0,Seated Forward Fold,0.444239,0.316826,-0.257815,0.999952,0.459517,0.279939,-0.234613,0.999941,0.463569,...,-0.293540,0.943346,0.832867,0.565701,0.008420,0.449771,0.920399,0.602261,-0.456241,0.931889
1,Seated Forward Fold,0.442934,0.316624,-0.237045,0.999948,0.456947,0.277399,-0.215936,0.999935,0.461113,...,-0.226043,0.936246,0.838900,0.573205,-0.009661,0.415647,0.920898,0.601793,-0.393000,0.925020
2,Seated Forward Fold,0.438803,0.314242,-0.255944,0.999952,0.453962,0.275179,-0.231228,0.999940,0.458440,...,-0.242850,0.933787,0.837688,0.577000,-0.017843,0.390006,0.919807,0.605743,-0.404771,0.924056
3,Seated Forward Fold,0.438297,0.315396,-0.255016,0.999954,0.453978,0.276854,-0.230898,0.999942,0.458197,...,-0.249860,0.933799,0.835670,0.573054,-0.046737,0.372359,0.918191,0.608442,-0.405845,0.926051
4,Seated Forward Fold,0.438522,0.317592,-0.249539,0.999956,0.454647,0.279244,-0.221311,0.999945,0.458459,...,-0.234493,0.926729,0.862209,0.574616,-0.026964,0.343819,0.918044,0.597813,-0.399701,0.919846


In [5]:
# normalization function to make model robust to different positions and body sizes
def normalize_landmarks(row):
    # hip midpoint as origin
    hip_mid_x = (row['x23'] + row['x24']) / 2
    hip_mid_y = (row['y23'] + row['y24']) / 2
    hip_mid_z = (row['z23'] + row['z24']) / 2
    
    shoulder_mid_x = (row['x11'] + row['x12']) / 2
    shoulder_mid_y = (row['y11'] + row['y12']) / 2
    
    torso_size = np.sqrt((shoulder_mid_x - hip_mid_x)**2 + (shoulder_mid_y - hip_mid_y)**2)
    
    for i in range(33):
        row[f'x{i}'] = (row[f'x{i}'] - hip_mid_x) / torso_size
        row[f'y{i}'] = (row[f'y{i}'] - hip_mid_y) / torso_size
        row[f'z{i}'] = (row[f'z{i}'] - hip_mid_z) / torso_size
        
    return row


In [7]:
df_normalized = df.apply(normalize_landmarks, axis=1)
# Hip midpoint should now be close to 0
print(df_normalized[['x23', 'y23', 'z23', 'x24', 'y24', 'z24']].describe())

               x23          y23          z23          x24          y24  \
count  1842.000000  1842.000000  1842.000000  1842.000000  1842.000000   
mean      0.025724    -0.007265    -0.010402    -0.025724     0.007265   
std       0.041308     0.049883     0.382760     0.041308     0.049883   
min      -0.048454    -0.140466    -0.573643    -0.141007    -0.104425   
25%      -0.016695    -0.053327    -0.386392    -0.044850    -0.038812   
50%       0.032167    -0.010051    -0.274940    -0.032167     0.010051   
75%       0.044850     0.038812     0.359795     0.016695     0.053327   
max       0.141007     0.104425     0.555008     0.048454     0.140466   

               z24  
count  1842.000000  
mean      0.010402  
std       0.382760  
min      -0.555008  
25%      -0.359795  
50%       0.274940  
75%       0.386392  
max       0.573643  


In [9]:
# input into model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

X = df_normalized.drop('label', axis=1).values

le = LabelEncoder()
y = le.fit_transform(df_normalized['label'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=2026)

print(le.classes_)  # shows which index maps to which pose
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

["Child's Pose" 'Cobra Pose' 'Seated Forward Fold']
X_train: (1289, 132)
X_test: (553, 132)
y_train: (1289,)
y_test: (553,)


In [ ]:
# input into model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam


stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

model = Sequential()
model.add(tf.keras.Input(shape=(132, )))
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(len(le.classes_), activation='softmax'))

model.compile(optimizer=Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])


Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_24 (Dense)                │ (None, 128)            │        17,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,459 (107.26 KB)

 Trainable params: 27,459 (107.26 KB)

 Non-trainable params: 0 (0.00 B)

In [20]:
# Train the model
model.fit(X_train, y_train, batch_size=32, epochs=50, validation_split=0.1, callbacks=[stop_early])
# observe output of model on test set
y_conf= model.predict(X_test)
max_confidence = np.max(y_conf, axis=1)
y_pred = np.argmax(y_conf, axis=1)

y_pred_cat = []
for conf, pred in zip(max_confidence, y_pred):
    if conf < 0.5:
        y_pred_cat.append('Resting Position')
    else:
        y_pred_cat.append(le.classes_[pred])

Epoch 1/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.8284 - loss: 0.4470 - val_accuracy: 1.0000 - val_loss: 0.0205
Epoch 2/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9853 - loss: 0.0766 - val_accuracy: 1.0000 - val_loss: 0.0015
Epoch 3/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9957 - loss: 0.0322 - val_accuracy: 1.0000 - val_loss: 2.3905e-04
Epoch 4/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9948 - loss: 0.0189 - val_accuracy: 1.0000 - val_loss: 1.2522e-04
Epoch 5/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9966 - loss: 0.0156 - val_accuracy: 1.0000 - val_loss: 4.1935e-05
Epoch 6/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9991 - loss: 0.0067 - val_accuracy: 1.0000 - val_loss: 2.4075e-05
Epoch 7/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9991 - loss: 0.0060 - val_accuracy: 1.0000 - val_loss: 7.6311e-06
Epoch 8/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9974 - loss: 0.0084 - val_accurac

In [21]:
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       175
           1       1.00      1.00      1.00       199
           2       1.00      1.00      1.00       179

    accuracy                           1.00       553
   macro avg       1.00      1.00      1.00       553
weighted avg       1.00      1.00      1.00       553

[[175   0   0]
 [  0 199   0]
 [  0   0 179]]


In [22]:
import json

model.save('yoga_pose_classifier.keras', overwrite=True)
config = {
    'resting_threshold': 0.5,
    'label_encoder': le.classes_.tolist()
    }
with open('model_config.json', 'w') as f:
    json.dump(config, f)